In [47]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA


In [20]:
df = pd.read_csv("DateFruit_Dataset.csv")

In [21]:
df.head() # 898 samples, 34 features, 1 category

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [22]:
X = df.drop("Class",axis=1)
y = df["Class"]

In [23]:
df["Class"].unique() # 7 neurons in o/p layer

<StringArray>
['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY']
Length: 7, dtype: str

In [24]:
le = LabelEncoder()
y = le.fit_transform(y)


In [25]:
X_train, X_test, y_train,y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

In [26]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

### ANN

In [28]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

In [31]:
X_train_tensor = torch.tensor(X_train_scaled, dtype = torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype = torch.float32)
y_train_tensor = torch.tensor(y_train, dtype = torch.long)
y_test_tensor = torch.tensor(y_test, dtype = torch.long)

In [32]:
train_dataset = TensorDataset(X_train_tensor,y_train_tensor)
test_dataset = TensorDataset(X_test_tensor,y_test_tensor)


In [33]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32)

### Build Our Model

In [34]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()

        self.model = nn.Sequential(
            nn.Linear(X.shape[1], 64),
            nn.ReLU(),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Linear(64,7)
            )

    def forward(self,x):
        return self.model(x)

In [36]:
model = ANN()

# loss & optimizer

criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [38]:
# Training the NN

# not validation becz data is small

epochs = 100
for epoch in range(epochs):
    model.train()

    running_loss=0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()
        
        outputs = model(xb)
        loss = criteria(outputs,yb)
        loss.backward()
        optimizer.step()  # params update

        running_loss +=loss.item() #.item cz its a tensor value and we converting it to float

    train_loss = running_loss/len(train_loader)

    print(f"epoch = {epoch+1}/{epochs}, loss ={train_loss} ")

epoch = 1/100, loss =0.02346080153897081 
epoch = 2/100, loss =0.023895208379658667 
epoch = 3/100, loss =0.02795790198623486 
epoch = 4/100, loss =0.02648634690305461 
epoch = 5/100, loss =0.023634379962459207 
epoch = 6/100, loss =0.021278209336425945 
epoch = 7/100, loss =0.025893428542590496 
epoch = 8/100, loss =0.02346929839974188 
epoch = 9/100, loss =0.01920005054537045 
epoch = 10/100, loss =0.01988367563235047 
epoch = 11/100, loss =0.019238657101674977 
epoch = 12/100, loss =0.01794466198138569 
epoch = 13/100, loss =0.018796489978938 
epoch = 14/100, loss =0.01747420727801712 
epoch = 15/100, loss =0.01927461967115169 
epoch = 16/100, loss =0.016630153768979337 
epoch = 17/100, loss =0.016454375872854143 
epoch = 18/100, loss =0.018218365806640813 
epoch = 19/100, loss =0.017023832702999123 
epoch = 20/100, loss =0.016493971368459905 
epoch = 21/100, loss =0.013982008418837644 
epoch = 22/100, loss =0.018641704794667337 
epoch = 23/100, loss =0.013885333630479063 
epoch = 2

In [44]:
# Evaluate

model.eval()

total = 0 # total values in test data
correct = 0 # how much correct calculated

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb) # [0.2,0.5,1.3,-0.5]
        _ , predicted = torch.max(outputs,1) # torch.max gives the max value and its index, _ -> is value we don't wnt to save

        correct += (predicted==yb).sum().item()
        total+= yb.size(0) # actual samples in each batch

print("total values: ", total)
print("correct values: ", correct)
print("accuracy: ", correct/total*100)

total values:  180
correct values:  169
accuracy:  93.88888888888889


### With PCA


In [76]:
pca = PCA(n_components=15)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

In [77]:
X_train_tensor = torch.tensor(X_train_pca, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_pca, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [78]:
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [79]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(X_train_pca.shape[1], 64),
            nn.ReLU(),

            nn.Linear(64, 64),
            nn.ReLU(),

            nn.Linear(64, 7)
        )

    def forward(self, x):
        return self.model(x)

model = ANN()

criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

epochs = 100

for epoch in range(epochs):
    model.train()

    running_loss = 0.0

    for xb, yb in train_loader:
        optimizer.zero_grad()

        outputs = model(xb)
        loss = criteria(outputs, yb)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)

    print(f"epoch = {epoch+1}/{epochs}, loss = {train_loss}")

epoch = 1/100, loss = 1.6231682300567627
epoch = 2/100, loss = 1.0899322551229726
epoch = 3/100, loss = 0.7496866117353025
epoch = 4/100, loss = 0.5549517766289089
epoch = 5/100, loss = 0.456284471180128
epoch = 6/100, loss = 0.38307592272758484
epoch = 7/100, loss = 0.3344872464304385
epoch = 8/100, loss = 0.2911187494578569
epoch = 9/100, loss = 0.2671199385886607
epoch = 10/100, loss = 0.24034075957277548
epoch = 11/100, loss = 0.22618019645628723
epoch = 12/100, loss = 0.21535389935192856
epoch = 13/100, loss = 0.19721352892077487
epoch = 14/100, loss = 0.1895221568967985
epoch = 15/100, loss = 0.17324549640002457
epoch = 16/100, loss = 0.1686893225364063
epoch = 17/100, loss = 0.16148235195356866
epoch = 18/100, loss = 0.16178100280787633
epoch = 19/100, loss = 0.15209664145241614
epoch = 20/100, loss = 0.14996367271827615
epoch = 21/100, loss = 0.1457484952457573
epoch = 22/100, loss = 0.13805037720695787
epoch = 23/100, loss = 0.1319306713083516
epoch = 24/100, loss = 0.12367268

In [80]:
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)
        _, predicted = torch.max(outputs, 1)

        correct += (predicted == yb).sum().item()
        total += yb.size(0)

print("total values:", total)
print("correct values:", correct)
print("accuracy:", correct/total*100)

total values: 180
correct values: 170
accuracy: 94.44444444444444


In [81]:
print(pca.explained_variance_ratio_)
print("Total variance retained:",
      pca.explained_variance_ratio_.sum())
print("Variance retained:",
      pca.explained_variance_ratio_.sum())

[0.40992337 0.22985566 0.11387629 0.06257129 0.04991463 0.03679119
 0.02674211 0.01828489 0.014797   0.00793891 0.00645975 0.00483848
 0.00400066 0.00340697 0.0027039 ]
Total variance retained: 0.99210511032433
Variance retained: 0.99210511032433
